# Black–Scholes benchmarks vs Monte Carlo

Closed-form **Black–Scholes** prices anchor **European** Monte Carlo. Use them for **put–call parity** checks, **convergence** in **`num_paths`**, and **finite-difference Greeks** as a teaching reference.

## Concept

For GBM with constant volatility, **`black_scholes_call`** / **`black_scholes_put`** implement the standard formulas. **Put–call parity** links calls and puts with the same strike and expiry. **Monte Carlo error** scales like \( 1/\sqrt{N} \) for crude averaging. **Bump-and-revalue** on spot and vol approximates **delta** and **vega**.

## API walkthrough

Import **`black_scholes_call`**, **`black_scholes_put`**, and **`EuropeanPricer`**. Compare **`MonteCarloResult.mean.amount`** to analytical prices.

In [ ]:
import math
from finstack.monte_carlo import black_scholes_call, black_scholes_put, EuropeanPricer

S, K, r, q, vol, T = 100.0, 100.0, 0.05, 0.02, 0.25, 1.0
c = black_scholes_call(S, K, r, q, vol, T)
p = black_scholes_put(S, K, r, q, vol, T)
print(f"BS call: {c:.8f}")
print(f"BS put:  {p:.8f}")

forward = S * math.exp((r - q) * T)
parity_lhs = c - p
parity_rhs = math.exp(-r * T) * (forward - K)
print(f"Put-call parity LHS (C-P): {parity_lhs:.8f}")
print(f"Put-call parity RHS:       {parity_rhs:.8f}")
print(f"Parity residual:           {abs(parity_lhs - parity_rhs):.2e}")


## Examples

**Convergence study**: increase **`num_paths`** and tabulate **absolute error** versus **`black_scholes_call`**.

**Greeks (FD)**: central differences on **`black_scholes_call`** for **spot** (delta) and **vol** (vega).

In [ ]:
from finstack.monte_carlo import black_scholes_call, EuropeanPricer

S, K, r, q, vol, T = 100.0, 100.0, 0.05, 0.0, 0.20, 1.0
analytical = black_scholes_call(S, K, r, q, vol, T)
print(f"Analytical ATM call: {analytical:.8f}")
counts = [1000, 5000, 10_000, 50_000, 100_000]
print("num_paths | MC_mean | stderr | |err|")
for n in counts:
    est = EuropeanPricer(num_paths=n, seed=123).price_call(S, K, r, q, vol, T, num_steps=252)
    m = est.mean.amount
    print(f"{n:9d} | {m:.8f} | {est.stderr:.8f} | {abs(m - analytical):.8f}")


In [ ]:
from finstack.monte_carlo import black_scholes_call

S, K, r, q, vol, T = 100.0, 100.0, 0.05, 0.0, 0.20, 1.0
h_s = 1.0
delta = (
    black_scholes_call(S + h_s, K, r, q, vol, T)
    - black_scholes_call(S - h_s, K, r, q, vol, T)
) / (2 * h_s)
print(f"Finite-difference delta (central, bump spot={h_s}): {delta:.8f}")

h_v = 0.01
vega = (
    black_scholes_call(S, K, r, q, vol + h_v, T)
    - black_scholes_call(S, K, r, q, vol - h_v, T)
) / (2 * h_v)
print(f"Finite-difference vega (central, bump vol={h_v}): {vega:.8f}")
print("(Vega here is per absolute vol bump; scale to 'per 1% vol' by dividing by 100 if needed.)")


## Takeaways

- **Analytical BS** is the first sanity check for **`EuropeanPricer`**.
- **Put–call parity** catches discounting or input mismatches instantly.
- **Error vs paths** illustrates **\( 1/\sqrt{N} \)** behavior of standard errors.
- **Bumping** is simple and robust for teaching; production often prefers **pathwise** or **likelihood ratio** Greeks for MC.